# Módulo I: Semana 2 - Modulación Digital Avanzada y Eficiencia Espectral
## Medios y Comunicaciones Digitales
## Miguel Ángel Walle Vázquez
## 2026-02-06

### 1. Contextualización

**Concepto Central:** El espectro radioeléctrico es un recurso finito y costoso. El objetivo del ingeniero no es solo transmitir, sino maximizar la **Eficiencia Espectral ($\eta$)**.

* **Métrica:** $\eta = \frac{\text{Tasa de bits (bps)}}{\text{Ancho de banda (Hz)}}$.
* **Ejemplo Real:** ¿Cómo logramos que Wi-Fi 7 sea más rápido que Wi-Fi 6 si usamos el mismo aire? La respuesta es aumentando la densidad de bits por símbolo (de 1024-QAM a 4096-QAM).

---


### 2. Desarrollo Teórico
#### 2.1. El Espacio de Señales e $I/Q$

Para transmitir múltiples bits en un solo pulso, manipulamos dos portadoras sinusoidales ortogonales: una en fase ($\cos$) y otra en cuadratura ($\sin$).

* **Ecuación Fundamental:**

$$s(t) = A_I(t)\cos(2\pi f_c t) - A_Q(t)\sin(2\pi f_c t)$$

* **Explicación:** Cada par de amplitudes $(A_I, A_Q)$ define un punto en un plano cartesiano llamado **Constelación**.
* **Ejemplo:** En **16-QAM**, tenemos 16 combinaciones posibles. Como $16 = 2^4$, cada punto representa un grupo de **4 bits**.

#### 2.2. Optimización mediante Mapeo de Gray

**Problema:** Si el ruido altera un símbolo, el receptor podría confundirlo con un vecino. Si el diseño es malo, un error de un símbolo podría causar que todos los bits del grupo fallen.

* **Solución:** El **Mapeo de Gray** asegura que los símbolos adyacentes difieran en **solo 1 bit**.


* **Estrategia Pedagógica:** Revisar la imagen anexa en la práctica, que ilustra una constelación 4-QAM comparando un mapeo binario estándar (00, 01, 10, 11) donde los vecinos arriba/abajo cambian 2 bits, frente a Gray (00, 01, 11, 10).



#### 2.3. La Batalla contra el Ruido (Análisis AWGN)

La probabilidad de error ($P_e$) depende de la distancia entre los puntos de la constelación y la potencia del ruido.

* **Fórmula de Ingeniería:**

$$P_e \approx 4 \left( 1 - \frac{1}{\sqrt{M}} \right) Q \left( \sqrt{\frac{3 k E_b}{(M-1)N_0}} \right)$$


* **Interpretación:**
* A mayor $M$ (densidad), los puntos están más cerca y la $P_e$ sube.
* La función $Q$ representa la probabilidad de que el ruido sea tan fuerte que "empuje" el punto más allá del umbral de decisión.

#### 2.4. El Cambio de Paradigma: OFDM

En canales de alta velocidad, una sola portadora ancha sufre de "desvanecimiento selectivo" (algunas frecuencias se pierden, arruinando todo el mensaje).

* **Solución:** **OFDM (Orthogonal Frequency Division Multiplexing)** divide el canal en $K$ subportadoras ortogonales estrechas.
* **Matemáticas Digitales:** El transmisor usa la **IFFT** (Transformada Rápida de Fourier Inversa) para convertir los bits del dominio de la frecuencia al tiempo:

$$x[n] = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} X[k] e^{j \frac{2\pi nk}{N}}$$

---

### 3. Ejemplos de Aplicación y Discusión

1. **Cálculo de Bits:**
* *Pregunta:* Si un sistema utiliza 1024-QAM, ¿cuántos bits transmite por símbolo?
* *Solución:* $k = \log_2(1024) = 10 \text{ bits/símbolo}$.

2. **Robustez vs. Velocidad:**
* *Escenario:* Un usuario de celular se aleja de la antena. El ruido sube.
* *Acción del Ingeniero:* El sistema cambia automáticamente de 256-QAM (rápido pero frágil) a QPSK (lento pero robusto). Esto se llama Modulación Adaptativa.
---

### 4. Práctica: Laboratorio en Python

Se entregan dos scripts base (anexo a esta práctica) para validar la teoría.

#### Ejercicio 1: Simulación de Curvas BER (Bit Error Rate)

El objetivo es graficar la "cascada de error" para 16-QAM y ver cómo el ruido destruye la información.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parámetros: Eb/N0 en dB
eb_no_db = np.arange(0, 15, 2)
# Simulación simplificada de la curva teórica de 16-QAM
ber_teorico = 0.5 * (3/4) * np.exp(-1.5 * (10**(eb_no_db/10)) / (15))

plt.semilogy(eb_no_db, ber_teorico, 'o-', label='16-QAM Teórico')
plt.grid(True)
plt.xlabel('Eb/N0 (dB)')
plt.ylabel('Tasa de Error de Bit (BER)')
plt.title('Validación de Desempeño en Canal AWGN')
plt.show()

#### Ejercicio B: Generación de Trama OFDM y Prefijo Cíclico

Demostrar cómo se protege la señal contra la interferencia de eco (ISI).

In [ ]:
K = 64  # Número de subportadoras
CP = K // 4  # Prefijo Cíclico (25%)

# 1. Generar símbolos aleatorios en frecuencia
simbolos_f = np.random.randn(K) + 1j*np.random.randn(K)
# 2. Pasar al dominio del tiempo (IFFT)
senal_tiempo = np.fft.ifft(simbolos_f)
# 3. Insertar Prefijo Cíclico (copiar el final al principio)
senal_con_cp = np.concatenate([senal_tiempo[-CP:], senal_tiempo])

print(f"Longitud original: {len(senal_tiempo)}, Con CP: {len(senal_con_cp)}")

---

### 5. Conclusiones y Cierre

* **Resumen:** M-QAM incrementa la velocidad pero exige mejor señal; OFDM combate el desvanecimiento del canal real.

---

### Referencias Bibliográficas

* Audinate. (2021). *Dante Certification Level 2: Advanced Networking and QoS*.
* Mathuranathan, V. (2019). *Digital Modulations using Python*. Silicon Press.
* Proakis, J. G., & Salehi, M. (2018). *Digital Communications* (5th ed.). McGraw-Hill.
* Sklar, B. (2001). *Digital Communications: Fundamentals and Applications* (2nd ed.). Prentice Hall.
* Telecom Infra Project (TIP). (2022). *Gaussian Noise Model in Optical Networking (GNPy documentation)*.